# 🐍 Python for Everybody — Course 3: Using Python to Access Web Data

> **Platform:** Coursera | **Instructor:** Dr. Charles Severance (University of Michigan)  
> **Level:** Beginner-Intermediate | **Language:** Python 3

---

## 📋 About This Course

This course teaches how to retrieve and process data from the internet. It covers regular expressions for pattern matching, raw socket connections, web scraping with BeautifulSoup, and parsing structured data formats like XML and JSON — skills at the core of data engineering and web automation.

---

## 📚 Table of Contents

| Week | Topic | Concepts |
|------|-------|----------|
| Week 2 | Regular Expressions | `re.findall()`, pattern matching, number extraction |
| Week 3 | Networking & Sockets | `socket`, HTTP GET, raw TCP connection |
| Week 4 (1) | Web Scraping — HTML | `urllib`, `BeautifulSoup`, `<span>` tags, sum |
| Week 4 (2) | Web Scraping — Link Following | `<a>` tags, iterative crawling, nth link |
| Week 5 | Parsing XML | `xml.etree.ElementTree`, `findall()`, data extraction |
| Week 6 (1) | Parsing JSON | `json.loads()`, nested dicts/lists, counting |
| Week 6 (2) | APIs & GeoData | Google Maps API, `urllib.parse`, JSON traversal |
| Bonus | GeoJSON Script | Full geocoding pipeline with coordinates |

---

## 📌 Week 2 — Regular Expressions

### Assignment Description
Open `regex_sum.txt`, use a regular expression to find all numbers in each line, and compute their total sum.

### Concepts Covered
- **`import re`** — Python's regular expression library
- **`re.findall(pattern, string)`** — find all matches of a pattern in a string
- **Pattern `[0-9]+`** — match one or more consecutive digits
- **Accumulator** — summing all extracted numbers

### Key Takeaway
> Regular expressions are a powerful mini-language for searching and extracting patterns in text. `[0-9]+` means "one or more digits" — a pattern used constantly in data cleaning and log parsing.

> 📁 **Required file:** `regex_sum.txt` (included in this course folder)

In [ ]:
# Week 2 — Sum all numbers in a file using regex
import re

handle = open('regex_sum.txt')
suma = 0

for line in handle:
    line.rstrip()
    y = re.findall('[0-9]+', line)
    for s in y:
        x = float(s)
        suma = suma + x

print(suma)

## 📌 Week 3 — Networking & Raw Sockets

### Assignment Description
Open a raw TCP socket connection to a web server, send an HTTP GET request manually, receive the response in chunks, and print it.

### Concepts Covered
- **`socket`** — Python's low-level networking library
- **`socket.AF_INET, SOCK_STREAM`** — IPv4 TCP socket
- **HTTP GET request** — the raw text format of a web request
- **`.recv(512)`** — receiving data in 512-byte chunks
- **`.encode()` / `.decode()`** — converting between bytes and strings

### Key Takeaway
> Every web browser does exactly this under the hood. Understanding raw sockets demystifies HTTP and shows what `urllib` abstracts away for us.

In [ ]:
# Week 3 — Raw socket HTTP GET request
import socket

mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect(('data.pr4e.org', 80))
cmd = 'GET http://data.pr4e.org/intro-short.txt HTTP/1.0\r\n\r\n'.encode()
mysock.send(cmd)

while True:
    data = mysock.recv(512)
    if len(data) < 1:
        break
    print(data.decode(), end='')

mysock.close()

## 📌 Week 4 (Assignment 1) — Web Scraping HTML with BeautifulSoup

### Assignment Description
Given a URL to an HTML page, extract all `<span>` tag values, print their details, and compute their sum.

**Example URL:** `http://py4e-data.dr-chuck.net/comments_703284.html`

### Concepts Covered
- **`urllib.request.urlopen()`** — fetching a web page
- **`BeautifulSoup`** — parsing HTML structure
- **`soup('span')`** — finding all `<span>` elements
- **`tag.contents[0]`** — extracting text inside a tag
- **SSL context** — bypassing certificate verification for educational URLs

### Key Takeaway
> BeautifulSoup turns messy HTML into a navigable tree. Finding tags by name (`soup('span')`) and reading their contents is the foundation of web scraping.

> 📦 **Requires:** `pip install beautifulsoup4`

In [ ]:
# Week 4 (1) — Scrape <span> tags and sum their values
from urllib.request import urlopen
from bs4 import BeautifulSoup
import ssl

# Ignore SSL certificate errors (for educational URLs)
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter URL: ')  # e.g. http://py4e-data.dr-chuck.net/comments_703284.html
html = urlopen(url, context=ctx).read()
soup = BeautifulSoup(html, 'html.parser')

# Find all <span> tags in the page
tags = soup('span')
total = 0

for tag in tags:
    print('TAG:', tag)
    print('URL:', tag.get('href', None))
    print('Attrs:', tag.attrs)
    print('Contents:', tag.contents[0])
    total = total + int(tag.contents[0])

print('Sum:', total)

## 📌 Week 4 (Assignment 2) — Link Following / Web Crawler

### Assignment Description
Starting from a URL, repeatedly follow the **18th link** on each page for 7 iterations and print the URL and name found at each step.

**Example URL:** `http://py4e-data.dr-chuck.net/known_by_Heather.html`

### Concepts Covered
- **`soup('a')`** — finding all anchor/link tags
- **`tag.get('href')`** — extracting the URL from a link
- **Iterative crawling** — using the output of one request as input for the next
- **Counter-based selection** — picking the nth element from a list

### Key Takeaway
> This is the basic logic behind a web crawler. By following links programmatically we can traverse entire websites — the foundation of how search engines work.

In [ ]:
# Week 4 (2) — Follow the 18th link on a page for 7 iterations
from urllib.request import urlopen
from bs4 import BeautifulSoup
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter URL: ')  # e.g. http://py4e-data.dr-chuck.net/known_by_Heather.html
html = urlopen(url, context=ctx).read()
soup = BeautifulSoup(html, 'html.parser')
tags = soup('a')

for x in range(1, 8):  # Follow link 7 times
    counter = 0
    for tag in tags:
        counter = counter + 1
        if counter == 18:  # Pick the 18th link
            print('URL:', tag.get('href', None))
            print('Contents:', tag.contents[0])
            url = tag.get('href', None)
            html = urlopen(url, context=ctx).read()
            soup = BeautifulSoup(html, 'html.parser')
            tags = soup('a')

## 📌 Week 5 — Parsing XML Data

### Assignment Description
Fetch an XML document from a URL, parse it with Python's XML library, count the entries, and sum a numeric field across all records.

**Example URL:** `http://py4e-data.dr-chuck.net/comments_703286.xml`

### Concepts Covered
- **`xml.etree.ElementTree`** — Python's built-in XML parser
- **`ET.fromstring()`** — parse XML from a string/bytes
- **`findall('path')`** — find all matching elements by XPath-like path
- **`item.find('tag').text`** — read the text content of a child element

### Key Takeaway
> XML is a tree of nested elements. `ElementTree` lets us traverse that tree using path expressions — essential for consuming APIs and data feeds that use XML format.

In [ ]:
# Week 5 — Parse XML from URL and sum values
from urllib.request import urlopen
import ssl
import xml.etree.ElementTree as ET

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter URL: ')  # e.g. http://py4e-data.dr-chuck.net/comments_703286.xml
html = urlopen(url, context=ctx).read()

tree = ET.fromstring(html)
lst = tree.findall('comments/comment')
print('Count:', len(lst))

total = 0
for item in lst:
    total = total + float(item.find('count').text)

print('Sum:', total)

## 📌 Week 6 (Assignment 1) — Parsing JSON Data

### Assignment Description
Fetch a JSON document from a fixed URL, parse it, and compute the sum of all `count` values inside the `comments` list.

**URL:** `http://py4e-data.dr-chuck.net/comments_703287.json`

### Concepts Covered
- **`json.loads()`** — parse a JSON string into Python dicts and lists
- **Nested data access** — navigating `js['comments'][n]['count']`
- **`range(N)`** — iterating a fixed number of times
- **`len(data)`** — checking response size

### Key Takeaway
> JSON maps directly to Python dictionaries and lists. Once parsed with `json.loads()`, you navigate it exactly like any other Python data structure — no special tools needed.

In [ ]:
# Week 6 (1) — Fetch JSON and sum 'count' values
import json
import urllib.request

url = 'http://py4e-data.dr-chuck.net/comments_703287.json'
uh = urllib.request.urlopen(url)
data = uh.read().decode()
print(len(data), 'characters retrieved')

js = json.loads(data)
comments = js['comments']
total = 0

for n in range(len(comments)):
    count = comments[n]['count']
    total = total + count

print('Sum:', total)

## 📌 Week 6 (Assignment 2) — Google Maps API & JSON Traversal

### Assignment Description
Call the Google Maps Geocoding API (or its course proxy) with a location name, parse the JSON response, and extract the `place_id` along with nested geographic data.

**Default location:** `University of West Florida`

### Concepts Covered
- **REST APIs** — sending parameters via URL query strings
- **`urllib.parse.urlencode()`** — encoding a dict as URL parameters
- **Deep JSON traversal** — accessing `js['results'][0]['geometry']['location']['lat']`
- **`try/except`** — handling malformed JSON responses gracefully
- **Infinite input loop** — continuing until the user exits

### Key Takeaway
> Real APIs return deeply nested JSON. The skill of navigating `dict['key'][index]['nested_key']` chains is fundamental to working with any modern web API.

In [ ]:
# Week 6 (2) — Google Maps Geocoding API
import urllib.request, urllib.parse, urllib.error
import json
import ssl

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

api_key = False
# Uncomment and add your key to use Google's real API:
# api_key = 'YOUR_GOOGLE_API_KEY'

if api_key is False:
    api_key = 42
    serviceurl = 'http://py4e-data.dr-chuck.net/json?'  # Course proxy
else:
    serviceurl = 'https://maps.googleapis.com/maps/api/geocode/json?'

while True:
    address = input('Enter location: ')
    if len(address) < 1:
        address = 'University of West Florida'

    parms = {'address': address}
    if api_key is not False:
        parms['key'] = api_key

    url = serviceurl + urllib.parse.urlencode(parms)
    print('Retrieving', url)

    uh = urllib.request.urlopen(url, context=ctx)
    data = uh.read().decode()
    print('Retrieved', len(data), 'characters')

    try:
        js = json.loads(data)
    except:
        js = None

    if not js or 'status' not in js or js['status'] != 'OK':
        print('==== Failure To Retrieve ====')
        print(data)
        continue

    print(json.dumps(js, indent=4))
    print('place_id:', js['results'][0]['place_id'])
    # Example of deep traversal:
    # print(js['results'][0]['geometry']['location']['lng'])
    # print(js['results'][0]['types'][1])
    break  # Remove this to keep asking for locations

## 📌 Bonus — Full GeoJSON Pipeline

### Description
An extended version of the geocoding script that also extracts and prints the latitude, longitude, and formatted address from the API response — a complete geocoding pipeline.

### Extra Concepts
- **Extracting coordinates** — `js['results'][0]['geometry']['location']['lat']`
- **Formatted address** — `js['results'][0]['formatted_address']`
- **`break` on empty input** — clean loop exit when user presses Enter

In [ ]:
# Bonus — Full geocoding pipeline with lat/lng extraction
import urllib.request, urllib.parse, urllib.error
import json
import ssl

api_key = False
if api_key is False:
    api_key = 42
    serviceurl = 'http://py4e-data.dr-chuck.net/json?'
else:
    serviceurl = 'https://maps.googleapis.com/maps/api/geocode/json?'

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

while True:
    address = input('Enter location (or press Enter to quit): ')
    if len(address) < 1: break

    parms = {'address': address}
    if api_key is not False: parms['key'] = api_key
    url = serviceurl + urllib.parse.urlencode(parms)

    print('Retrieving', url)
    uh = urllib.request.urlopen(url, context=ctx)
    data = uh.read().decode()
    print('Retrieved', len(data), 'characters')

    try:
        js = json.loads(data)
    except:
        js = None

    if not js or 'status' not in js or js['status'] != 'OK':
        print('==== Failure To Retrieve ====')
        print(data)
        continue

    print(json.dumps(js, indent=4))
    lat = js['results'][0]['geometry']['location']['lat']
    lng = js['results'][0]['geometry']['location']['lng']
    location = js['results'][0]['formatted_address']
    print(f'Lat: {lat}  |  Lng: {lng}')
    print('Address:', location)

---

## ✅ Course Summary

| Concept | Used In |
|---|---|
| Regular expressions (`re.findall`) | Week 2 |
| Raw sockets & HTTP | Week 3 |
| Web scraping with BeautifulSoup | Week 4 |
| XML parsing (`ElementTree`) | Week 5 |
| JSON parsing (`json.loads`) | Week 6 |
| REST API consumption | Week 6 (2), Bonus |
| Deep nested data traversal | Week 6 (2), Bonus |

> 🎓 **Certificate:** [Python for Everybody Specialization — Coursera](https://www.coursera.org/specializations/python)